In [1]:
# --------------------------------------------------
# Project Root
# --------------------------------------------------

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\IITG\Projects\audio_factor_disentanglement_v2


In [2]:
# --------------------------------------------------
# Imports
# --------------------------------------------------

import json
import pandas as pd

from src.utils.config_loader import load_yaml
from src.utils.path_manager import PathManager

from src.preprocessing.dataset_audit import DatasetAuditor
from src.visualization.audit_plots import AuditPlots

In [3]:
# --------------------------------------------------
# Load Config
# --------------------------------------------------

CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "data_config.yaml"
)

cfg = load_yaml(CONFIG_PATH)

cfg

{'project_name': 'audio_factor_disentanglement_v2',
 'audio_extensions': ['.wav', '.ogg', '.m4a'],
 'dataset': {'raw_dir': 'data/raw',
  'processed_dir': 'data/processed',
  'metadata_dir': 'data/metadata',
  'fragment_dir': 'data/fragments',
  'feature_dir': 'data/features'},
 'outputs': {'figures_dir': 'outputs/figures',
  'checkpoint_dir': 'outputs/checkpoints',
  'swap_dir': 'outputs/swaps'},
 'audio': {'target_sr': 16000, 'mono': True}}

In [4]:
# --------------------------------------------------
# Setup Paths
# --------------------------------------------------

pm = PathManager(
    project_root=PROJECT_ROOT,
    config=cfg
)

pm.create_required_dirs()

RAW_DIR = pm.get_path(
    cfg["dataset"]["raw_dir"]
)

METADATA_DIR = pm.get_path(
    cfg["dataset"]["metadata_dir"]
)

FIGURE_DIR = (
    pm.get_path(
        cfg["outputs"]["figures_dir"]
    )
    / "dataset_audit"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(RAW_DIR)
print(METADATA_DIR)
print(FIGURE_DIR)

d:\IITG\Projects\audio_factor_disentanglement_v2\data\raw
d:\IITG\Projects\audio_factor_disentanglement_v2\data\metadata
d:\IITG\Projects\audio_factor_disentanglement_v2\outputs\figures\dataset_audit


In [5]:
# --------------------------------------------------
# Dataset Audit
# --------------------------------------------------

auditor = DatasetAuditor(
    raw_dir=RAW_DIR,
    extensions=cfg["audio_extensions"]
)

df = auditor.build_inventory()

print(df.shape)

df.head()


Found 34 audio files



  0%|          | 0/34 [00:00<?, ?it/s]

c:\Users\Dell\.conda\envs\betavae\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\IITG\Projects\audio_factor_disentanglement_v2\src\utils\audio_loader.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
c:\Users\Dell\.conda\envs\betavae\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  3%|▎         | 1/34 [00:01<00:58,  1.79s/it]d:\IITG\Projects\audio_factor_disentanglement_v2\src\utils\audio_loader.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
c:\Users\Dell\.conda\envs\betavae\Lib\site-packages\librosa\core\audio.py:184: FutureWarning

(34, 12)


,filepath,filename,split,speaker,condition,extension,sample_rate,channels,duration_sec,num_samples,rms,peak
0,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_01.m4a,train,s1,clean,.m4a,48000,2,2.090667,100352,0.043963,0.314957
1,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_02.m4a,train,s1,clean,.m4a,48000,2,2.133333,102400,0.044887,0.301422
2,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_03.m4a,train,s1,clean,.m4a,48000,2,1.920000,92160,0.052497,0.321243
3,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_04.m4a,train,s1,clean,.m4a,48000,2,1.962667,94208,0.047852,0.279785
4,d:\IITG\Projects\audio_factor_disentanglement_...,s1_clean_05.m4a,train,s1,clean,.m4a,48000,2,2.133333,102400,0.050404,0.328415


In [6]:
# --------------------------------------------------
# Save Inventory
# --------------------------------------------------

csv_path = (
    METADATA_DIR
    / "audio_inventory.csv"
)

json_path = (
    METADATA_DIR
    / "audio_inventory.json"
)

df.to_csv(
    csv_path,
    index=False
)

df.to_json(
    json_path,
    orient="records",
    indent=4
)

print(csv_path)
print(json_path)

d:\IITG\Projects\audio_factor_disentanglement_v2\data\metadata\audio_inventory.csv
d:\IITG\Projects\audio_factor_disentanglement_v2\data\metadata\audio_inventory.json


In [7]:
# --------------------------------------------------
# Quick Dataset Summary
# --------------------------------------------------

display(
    df.groupby(
        ["split", "speaker", "condition"]
    )
    .size()
    .reset_index(name="count")
)

,split,speaker,condition,count
0,test,s1,clean,1
1,test,s2,noisy,1
2,train,s1,clean,8
3,train,s1,noisy,8
4,train,s2,clean,8
5,train,s2,noisy,8


In [8]:
# --------------------------------------------------
# Plot Statistics
# --------------------------------------------------

plots = AuditPlots(
    save_dir=FIGURE_DIR
)

plots.duration_histogram(df)

plots.rms_histogram(df)

plots.speaker_condition_matrix(df)

print("Basic plots generated.")

Basic plots generated.


In [9]:
# --------------------------------------------------
# Waveform Gallery
# --------------------------------------------------

plots.waveform_gallery(
    df["filepath"].tolist(),
    max_files=12
)

print("Waveform gallery generated.")

d:\IITG\Projects\audio_factor_disentanglement_v2\src\visualization\audit_plots.py:105: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(
c:\Users\Dell\.conda\envs\betavae\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Waveform gallery generated.


In [10]:
# --------------------------------------------------
# Spectrogram Gallery
# --------------------------------------------------

plots.spectrogram_gallery(
    df["filepath"].tolist(),
    max_files=12
)

print("Spectrogram gallery generated.")

d:\IITG\Projects\audio_factor_disentanglement_v2\src\visualization\audit_plots.py:150: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(


Spectrogram gallery generated.


In [11]:
# --------------------------------------------------
# Duration Statistics
# --------------------------------------------------

display(
    df[
        [
            "duration_sec",
            "sample_rate",
            "rms",
            "peak"
        ]
    ].describe()
)

,duration_sec,sample_rate,rms,peak
count,34.000000,34.000000,34.000000,34.000000
mean,2.336789,32000.000000,0.089989,0.619028
std,0.359624,16240.615006,0.049280,0.365106
min,1.920000,16000.000000,0.033427,0.178329
25%,2.077812,16000.000000,0.042206,0.283272
50%,2.238240,32000.000000,0.083002,0.633667
75%,2.477026,48000.000000,0.137877,0.989975
max,3.337813,48000.000000,0.165740,1.000000


In [12]:
# --------------------------------------------------
# Sample Rate Check
# --------------------------------------------------

display(
    df["sample_rate"]
    .value_counts()
    .sort_index()
)

sample_rate
16000    17
48000    17
Name: count, dtype: int64

In [13]:
# --------------------------------------------------
# Extension Check
# --------------------------------------------------

display(
    df["extension"]
    .value_counts()
)

extension
.m4a    17
.ogg    17
Name: count, dtype: int64